# 🤖 AI Interview Coach — Google Colab Deployment

This notebook deploys the AI Interview Coach Gradio app directly in Colab.

**Run each cell in order.** The final cell launches the app and prints a
public URL you can share with anyone.

### Prerequisites
- A free API key from one of the supported providers:
  - [Groq](https://console.groq.com) (recommended — free tier)
  - [Google AI Studio](https://aistudio.google.com)
  - [OpenAI](https://platform.openai.com)
  - [Anthropic](https://console.anthropic.com)


## 1 · Install dependencies


In [ ]:
!pip install -q gradio>=4.0.0 litellm>=1.0.0 openai==2.28.0 httpx==0.28.1 gTTS>=2.5.0 python-dotenv>=1.0.0


## 2 · Set your API key (optional)

You can enter your API key directly in the app UI, **or** set it here so
it's pre-loaded. Using Colab Secrets (🔑 icon in the sidebar) is the
safest approach — add a secret named e.g. `GROQ_API_KEY` and toggle
"Notebook access" on.

Alternatively, uncomment and paste your key below.


In [ ]:
import os

# Option A — Colab Secrets (recommended)
try:
    from google.colab import userdata
    for key_name in ['GROQ_API_KEY', 'OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GOOGLE_API_KEY']:
        try:
            val = userdata.get(key_name)
            if val:
                os.environ[key_name] = val
                print(f'✅ {key_name} loaded from Colab Secrets')
        except Exception:
            pass
except ImportError:
    pass

# Option B — Paste directly (less secure)
# os.environ['GROQ_API_KEY'] = 'gsk_...'  # uncomment and fill in


## 3 · Write application source files

The cells below create each source file in the Colab runtime.


In [ ]:
%%writefile llm_client.py
import logging
import os
import tempfile
from typing import Optional
import litellm
from openai import OpenAI
from gtts import gTTS

logger = logging.getLogger(__name__)

PROVIDERS = {
    "groq": "groq/openai/gpt-oss-120b",
    "openai": "gpt-4o",
    "anthropic": "claude-3-5-sonnet-20241022",
    "gemini": "gemini/gemini-1.5-flash",
}

DEFAULT_PROVIDER = "groq"

_API_KEY_ENV_VARS = {
    "groq": "GROQ_API_KEY",
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
    "gemini": "GOOGLE_API_KEY",
}

_GROQ_TRANSCRIPTION_BASE_URL = "https://api.groq.com/openai/v1"
_TRANSCRIPTION_MODELS = {
    "openai": "whisper-1",
    "groq": "whisper-large-v3-turbo",
}


def get_available_providers() -> list[str]:
    """Return list of supported provider names (for UI dropdown)."""
    return list(PROVIDERS.keys())


def validate_provider(provider: str) -> bool:
    """Return True if provider is known, False otherwise."""
    return provider in PROVIDERS


def get_api_key(provider: str) -> str:
    """Return the API key for the given provider loaded from the environment."""
    env_var = _API_KEY_ENV_VARS.get(provider, "")
    return os.getenv(env_var, "") if env_var else ""


def transcribe_audio(
    audio_path: str,
    provider: str,
    api_key: Optional[str] = None,
) -> str:
    """
    Transcribe an audio file to text.

    Supports OpenAI and Groq (OpenAI-compatible audio transcription API).
    """
    if not audio_path:
        raise ValueError("No audio file provided for transcription.")
    if provider not in _TRANSCRIPTION_MODELS:
        raise ValueError(
            "Speech transcription currently supports 'groq' and 'openai' providers."
        )

    resolved_key = api_key or get_api_key(provider)
    if not resolved_key:
        env_var = _API_KEY_ENV_VARS[provider]
        raise ValueError(
            f"No API key found for provider '{provider}'. "
            f"Set {env_var} in your .env file or enter it manually."
        )

    model = _TRANSCRIPTION_MODELS[provider]
    client_kwargs = {"api_key": resolved_key}
    if provider == "groq":
        client_kwargs["base_url"] = _GROQ_TRANSCRIPTION_BASE_URL

    logger.info("Transcribing audio via provider=%s model=%s path=%s", provider, model, audio_path)
    try:
        client = OpenAI(**client_kwargs)
        with open(audio_path, "rb") as audio_file:
            response = client.audio.transcriptions.create(model=model, file=audio_file)
        transcript = (response.text or "").strip()
        if not transcript:
            raise RuntimeError("Transcription returned empty text.")
        logger.info("Transcription successful, length=%d chars", len(transcript))
        return transcript
    except Exception as exc:
        logger.error("Transcription failed for provider=%s: %s", provider, exc)
        raise RuntimeError(
            f"Speech transcription failed for provider '{provider}': {exc}"
        ) from exc


def synthesize_speech(text: str) -> str:
    """Generate speech audio (mp3) for a text string and return the output filepath."""
    cleaned = (text or "").strip()
    if not cleaned:
        raise ValueError("No text provided for speech synthesis.")

    logger.info("Synthesizing speech, text_length=%d chars", len(cleaned))
    try:
        tts = gTTS(text=cleaned, lang="en")
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as tmp:
            out_path = tmp.name
        tts.save(out_path)
        logger.info("Speech synthesis complete, output=%s", out_path)
        return out_path
    except Exception as exc:
        logger.error("Speech synthesis failed: %s", exc)
        raise RuntimeError(f"Speech synthesis failed: {exc}") from exc


def _chat_openai(
    messages: list[dict],
    model: str,
    api_key: str,
    temperature: float,
) -> str:
    """Call OpenAI directly to avoid provider-adapter issues in some runtimes."""
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    content = response.choices[0].message.content
    if not content:
        raise RuntimeError("OpenAI API returned an empty response.")
    return content


def chat(
    messages: list[dict],
    provider: str,
    api_key: Optional[str] = None,
    temperature: float = 0.9,
) -> str:
    """
    Call the specified LLM provider and return the assistant's reply as a string.

    Args:
        messages:    OpenAI-style message list, e.g. [{"role": "user", "content": "Hi"}]
        provider:    One of the keys in PROVIDERS (e.g. "groq", "openai")
        api_key:     API key for the chosen provider. When omitted or empty the
                     key is loaded automatically from the corresponding environment
                     variable defined in _API_KEY_ENV_VARS.
        temperature: Sampling temperature (default 0.9)

    Returns:
        The content string from the first response choice.

    Raises:
        ValueError:    If provider is not in PROVIDERS or no API key is available.
        RuntimeError:  If the API call fails for any reason.
    """
    if not validate_provider(provider):
        raise ValueError(
            f"Unknown provider '{provider}'. "
            f"Available providers: {get_available_providers()}"
        )

    resolved_key = api_key or get_api_key(provider)
    if not resolved_key:
        env_var = _API_KEY_ENV_VARS[provider]
        raise ValueError(
            f"No API key found for provider '{provider}'. "
            f"Set {env_var} in your .env file or enter it manually."
        )

    model = PROVIDERS[provider]

    logger.info("LLM chat request: provider=%s model=%s temperature=%.2f messages=%d", provider, model, temperature, len(messages))
    try:
        if provider == "openai":
            result = _chat_openai(messages, model, resolved_key, temperature)
            logger.info("LLM chat response: provider=%s length=%d chars", provider, len(result))
            return result

        response = litellm.completion(
            model=model,
            messages=messages,
            temperature=temperature,
            api_key=resolved_key,
        )
        content = response.choices[0].message.content
        logger.info("LLM chat response: provider=%s length=%d chars", provider, len(content or ""))
        return content
    except Exception as exc:
        logger.error("LLM chat failed: provider=%s model=%s error=%s", provider, model, exc)
        raise RuntimeError(
            f"LLM API call failed for provider '{provider}': {exc}"
        ) from exc


In [ ]:
%%writefile prompts.py
"""
Prompt templates for the AI interview coach.
"""

INTERVIEWER_SYSTEM_PROMPT = (
    "You are a professional technical interviewer conducting a structured job interview. "
    "Your style is direct, encouraging, and grounded — you ask focused, clear questions "
    "appropriate to the role and seniority level. You are supportive without being "
    "sycophantic, and constructively critical without being harsh. "
    "You rely strictly on the job description and the candidate's responses; "
    "you never fabricate facts, invent requirements, or make assumptions beyond what "
    "is explicitly stated. Your goal is to give the candidate a fair, realistic interview "
    "experience that helps them grow."
)


def analyze_jd_prompt(jd: str) -> str:
    """Return a prompt that asks the LLM to analyze a job description and output JSON."""
    return (
        "Analyze the following job description and return a JSON object with these fields:\n\n"
        "- role_title (string): the job title as written in the description\n"
        "- seniority (string): one of 'junior', 'mid', 'senior', 'staff', or 'lead'\n"
        "- domain (string): the primary engineering/technical domain (e.g. 'backend engineering')\n"
        "- key_skills (array of strings): the most important technical skills mentioned\n"
        "- num_questions (integer): total interview questions to ask, based on seniority:\n"
        "    junior → 6–8, mid → 8–10, senior/staff/lead → 10–12\n"
        "- question_breakdown (object): how to distribute questions across categories:\n"
        "    { \"intro\": <int>, \"technical\": <int>, \"behavioral\": <int> }\n"
        "  The three values must sum to num_questions. "
        "  Use roughly 1 intro, ~70% technical, ~20-25% behavioral.\n\n"
        "Rules:\n"
        "- Respond with ONLY valid JSON. No markdown, no code fences, no commentary.\n"
        "- The response must be directly parseable by json.loads().\n\n"
        "Example output shape (values are illustrative):\n"
        '{\n'
        '  "role_title": "Senior Software Engineer",\n'
        '  "seniority": "senior",\n'
        '  "domain": "backend engineering",\n'
        '  "key_skills": ["Python", "REST APIs", "PostgreSQL"],\n'
        '  "num_questions": 10,\n'
        '  "question_breakdown": {\n'
        '    "intro": 1,\n'
        '    "technical": 7,\n'
        '    "behavioral": 2\n'
        '  }\n'
        '}\n\n'
        f"Job Description:\n{jd}"
    )


def generate_question_prompt(
    jd: str,
    role_info: dict,
    question_number: int,
    conversation_history: list[dict],
) -> str:
    """
    Return a prompt that asks the LLM to generate the next interview question.

    Args:
        jd: The full job description text.
        role_info: Parsed role metadata from analyze_jd_prompt (role_title, seniority, etc.).
        question_number: 1-based index of the question about to be asked.
        conversation_history: List of {role, content} dicts representing prior exchanges.
    """
    breakdown = role_info.get("question_breakdown", {})
    intro_count = breakdown.get("intro", 1)
    technical_count = breakdown.get("technical", 0)
    behavioral_count = breakdown.get("behavioral", 0)
    intro_end = intro_count
    technical_end = intro_count + technical_count
    planned_total = intro_count + technical_count + behavioral_count

    if question_number <= intro_end:
        phase = "introductory/warm-up"
        phase_guidance = (
            "Ask a friendly opening question such as a brief self-introduction or "
            "background summary. Keep it welcoming and low-pressure."
        )
    elif question_number <= technical_end:
        phase = "core technical"
        phase_guidance = (
            "Ask a substantive technical question directly relevant to the role's key skills "
            "and the job description. Vary depth and topic from previous questions."
        )
    elif question_number <= planned_total:
        phase = "behavioral"
        phase_guidance = (
            "Ask a behavioral question using a situational framing (e.g., 'Tell me about a "
            "time when…' or 'How have you handled…'). Focus on teamwork, communication, "
            "or problem-solving as relevant to the role."
        )
    else:
        phase = "extended"
        phase_guidance = (
            "Continue the interview with a mix of technical and behavioral questions. "
            "Vary the topic and depth from previous questions. Draw on the job description "
            "for relevant areas not yet covered."
        )

    history_text = ""
    if conversation_history:
        lines = []
        for turn in conversation_history:
            label = "Interviewer" if turn["role"] == "assistant" else "Candidate"
            lines.append(f"{label}: {turn['content']}")
        history_text = "\n\nConversation so far:\n" + "\n\n".join(lines)

    return (
        f"You are interviewing a candidate for the role: {role_info.get('role_title', 'the position')} "
        f"({role_info.get('seniority', 'unknown')} level, {role_info.get('domain', 'technical')}).\n\n"
        f"This is question {question_number}. "
        f"Phase: {phase}.\n"
        f"Guidance: {phase_guidance}\n\n"
        f"Job Description:\n{jd}"
        f"{history_text}\n\n"
        "Instructions:\n"
        "- Output ONLY the question text. No preamble, no labels, no 'Sure! Here's question N:' intro.\n"
        "- The question must be concise, specific, and directly relevant to the job description.\n"
        "- Do not repeat a topic already covered in the conversation above.\n"
        "- Do not add any follow-up commentary after the question."
    )


def classify_response_prompt(
    question: str,
    candidate_response: str,
) -> str:
    """
    Return a prompt that asks the LLM to classify whether the candidate's
    response is a clarifying question or an actual answer.
    """
    return (
        "You are classifying a candidate's response during a technical interview.\n\n"
        f"Interview question: {question}\n\n"
        f"Candidate's response: {candidate_response}\n\n"
        "Is the candidate asking a clarifying question about the interview question, "
        "or are they providing their actual answer?\n\n"
        "Rules:\n"
        "- If the response is asking for more information, clarification, or context "
        "about the question, classify it as CLARIFYING.\n"
        "- If the response is attempting to answer the question (even partially or "
        "poorly), classify it as ANSWER.\n"
        "- Respond with ONLY one word: either CLARIFYING or ANSWER. Nothing else."
    )


def clarifying_question_prompt(
    question: str,
    clarifying_question: str,
    conversation_history: list[dict],
) -> str:
    """
    Return a prompt for the interviewer to respond to a candidate's clarifying question.

    The interviewer should answer helpfully without giving away the answer.
    """
    history_text = ""
    if conversation_history:
        lines = []
        for turn in conversation_history:
            label = "Interviewer" if turn["role"] == "assistant" else "Candidate"
            lines.append(f"{label}: {turn['content']}")
        history_text = "\n\nConversation so far:\n" + "\n\n".join(lines)

    return (
        "You are a professional technical interviewer. The candidate has asked a "
        "clarifying question about your most recent interview question.\n\n"
        f"Your interview question was: {question}\n\n"
        f"Candidate's clarifying question: {clarifying_question}\n"
        f"{history_text}\n\n"
        "Instructions:\n"
        "- Answer the clarifying question helpfully and concisely.\n"
        "- Provide enough context for the candidate to answer well, but do NOT "
        "give away the answer itself.\n"
        "- Stay in character as the interviewer.\n"
        "- Output ONLY your response. No labels or preamble."
    )


def generate_feedback_prompt(
    question: str,
    answer: str,
    question_number: int,
    resume: str = "",
) -> str:
    """
    Return a prompt asking the LLM to evaluate a candidate's answer.

    Args:
        question: The interview question that was asked.
        answer: The candidate's response.
        question_number: 1-based position of this question.
        resume: Optional candidate resume text for context.
    """
    resume_section = ""
    if resume.strip():
        resume_section = (
            f"\n\nCandidate's Resume (use this for additional context when evaluating):\n{resume}\n"
        )

    return (
        f"You are evaluating a candidate's answer to question {question_number} "
        "in a technical interview.\n\n"
        f"Question: {question}\n\n"
        f"Candidate's Answer: {answer}\n"
        f"{resume_section}\n"
        "Provide concise, specific feedback on this answer. "
        "Keep the total response to 3–6 sentences across all sections. "
        "Be constructive and concrete — reference what the candidate actually said.\n\n"
        "Format your response as structured markdown with exactly these four sections:\n\n"
        "**Strengths**\n"
        "- bullet points of what was good about the answer\n\n"
        "**Areas to Improve**\n"
        "- bullet points of what could be stronger or was missing\n\n"
        "**Score**: X/5 — one-line justification\n\n"
        "**Example Strong Answer**\n"
        "Write a concise example of how a strong candidate would answer this specific question, "
        "addressing the gaps identified above. Keep it to 3–5 sentences.\n\n"
        "Scoring guide:\n"
        "  5 – Excellent: thorough, accurate, well-structured\n"
        "  4 – Good: solid answer with minor gaps\n"
        "  3 – Adequate: covers the basics but lacks depth\n"
        "  2 – Weak: significant gaps or inaccuracies\n"
        "  1 – Poor: largely incorrect or off-topic\n\n"
        "Do not add any text before or after the four markdown sections."
    )


def generate_summary_prompt(
    jd: str,
    role_info: dict,
    qa_pairs: list[dict],
    resume: str = "",
) -> str:
    """
    Return a prompt for generating a final interview summary report.

    Args:
        jd: The full job description text.
        role_info: Parsed role metadata (role_title, seniority, domain, key_skills, etc.).
        qa_pairs: List of dicts with keys: question, answer, feedback, score.
        resume: Optional candidate resume text for context.
    """
    qa_text_lines = []
    for i, qa in enumerate(qa_pairs, start=1):
        qa_text_lines.append(
            f"Q{i}: {qa['question']}\n"
            f"Answer: {qa['answer']}\n"
            f"Feedback: {qa['feedback']}\n"
            f"Score: {qa.get('score', 'N/A')}/5"
        )
    qa_text = "\n\n".join(qa_text_lines)

    role_title = role_info.get("role_title", "the role")
    seniority = role_info.get("seniority", "")
    domain = role_info.get("domain", "")

    resume_section = ""
    if resume.strip():
        resume_section = f"\nCandidate's Resume:\n{resume}\n"

    return (
        f"You have just completed a mock interview for the role: {role_title} "
        f"({seniority}, {domain}).\n\n"
        f"Job Description:\n{jd}\n"
        f"{resume_section}\n"
        f"Interview Q&A with per-question feedback and scores:\n\n{qa_text}\n\n"
        "Generate a final interview summary report. "
        "Compute the Overall Score as the average of the per-question scores, rounded to one decimal place. "
        "Base all observations strictly on the candidate's actual answers above.\n\n"
        "Format your response as structured markdown with exactly these five sections:\n\n"
        "**Overall Score**: X.X/5\n\n"
        f"**Role**: {role_title}\n\n"
        "**Top Strengths**\n"
        "- (3 bullet points highlighting the candidate's strongest demonstrated skills/qualities)\n\n"
        "**Key Development Areas**\n"
        "- (3 bullet points identifying the most important areas for improvement)\n\n"
        "**Recommendation**: Hire / Consider / Pass — brief one-to-two sentence justification\n\n"
        "Recommendation guide:\n"
        "  Hire     – Overall score ≥ 4.0 and no critical gaps\n"
        "  Consider – Overall score 2.5–3.9 or notable strengths offset by gaps\n"
        "  Pass     – Overall score < 2.5 or significant skill misalignment\n\n"
        "Do not add any text before or after the five markdown sections."
    )


In [ ]:
%%writefile interview_agent.py
"""
Core interview state machine.

Coordinates prompts.py and llm_client.py to manage a full interview session.
"""

from __future__ import annotations

import json
import logging
import re
from dataclasses import dataclass, field

logger = logging.getLogger(__name__)

from llm_client import chat
from prompts import (
    INTERVIEWER_SYSTEM_PROMPT,
    analyze_jd_prompt,
    classify_response_prompt,
    clarifying_question_prompt,
    generate_feedback_prompt,
    generate_question_prompt,
    generate_summary_prompt,
)

_SCORE_RE = re.compile(r"\*{0,2}Score\*{0,2}:\s*([0-9]+(?:\.[0-9]+)?)\s*/\s*5", re.IGNORECASE)

_DEFAULT_ROLE_INFO = {
    "role_title": "Software Engineer",
    "seniority": "mid",
    "domain": "software engineering",
    "key_skills": [],
    "num_questions": 8,
    "question_breakdown": {"intro": 1, "technical": 5, "behavioral": 2},
}


@dataclass
class InterviewState:
    jd: str
    provider: str
    api_key: str
    role_info: dict
    resume: str = ""
    questions: list[str] = field(default_factory=list)
    answers: list[str] = field(default_factory=list)
    feedbacks: list[str] = field(default_factory=list)
    scores: list[float] = field(default_factory=list)
    conversation_history: list[dict] = field(default_factory=list)
    phase: str = "setup"
    current_index: int = 0


def _build_messages(user_prompt: str) -> list[dict]:
    return [
        {"role": "system", "content": INTERVIEWER_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


def _extract_score(feedback_text: str) -> float:
    match = _SCORE_RE.search(feedback_text)
    if match:
        return float(match.group(1))
    return 3.0


def start_interview(jd: str, provider: str, api_key: str, resume: str = "") -> tuple["InterviewState", str]:
    """
    Initialise a new interview session.

    Analyses the job description, creates an InterviewState, and returns the
    state along with the first interview question.
    """
    try:
        raw = chat(_build_messages(analyze_jd_prompt(jd)), provider, api_key, temperature=0.2)
    except Exception as exc:
        logger.error("JD analysis failed: provider=%s error=%s", provider, exc)
        raise RuntimeError(
            f"Failed to analyse the job description via {provider}: {exc}"
        ) from exc

    # Strip accidental markdown code fences the model may add despite instructions.
    cleaned = re.sub(r"^```[a-z]*\s*|\s*```$", "", raw.strip(), flags=re.MULTILINE)
    try:
        role_info = json.loads(cleaned)
        logger.info("JD analysis parsed successfully: role=%s seniority=%s",
                     role_info.get("role_title"), role_info.get("seniority"))
    except json.JSONDecodeError:
        logger.warning("JD analysis returned invalid JSON, using defaults. Raw response: %.200s", cleaned)
        role_info = dict(_DEFAULT_ROLE_INFO)

    # Ensure required keys exist with sensible fallbacks.
    role_info.setdefault("role_title", _DEFAULT_ROLE_INFO["role_title"])
    role_info.setdefault("seniority", _DEFAULT_ROLE_INFO["seniority"])
    role_info.setdefault("domain", _DEFAULT_ROLE_INFO["domain"])
    role_info.setdefault("key_skills", [])
    role_info.setdefault("num_questions", _DEFAULT_ROLE_INFO["num_questions"])
    role_info.setdefault("question_breakdown", dict(_DEFAULT_ROLE_INFO["question_breakdown"]))

    state = InterviewState(
        jd=jd,
        provider=provider,
        api_key=api_key,
        role_info=role_info,
        resume=resume,
        phase="interviewing",
        current_index=0,
    )

    logger.info("Interview started: provider=%s role=%s resume_provided=%s",
                provider, role_info.get("role_title"), bool(resume))

    first_question = _generate_next_question(state)
    return state, first_question


def _generate_next_question(state: InterviewState) -> str:
    """Ask the LLM for the next interview question, update state, and return it."""
    question_number = len(state.questions) + 1
    logger.info("Generating question %d", question_number)

    try:
        question_text = chat(
            _build_messages(
                generate_question_prompt(
                    state.jd,
                    state.role_info,
                    question_number,
                    state.conversation_history,
                )
            ),
            state.provider,
            state.api_key,
            temperature=0.9,
        )
    except Exception as exc:
        logger.error("Question generation failed: question_number=%d error=%s", question_number, exc)
        raise RuntimeError(
            f"Failed to generate question {question_number} via {state.provider}. "
            "Check your API key and try again."
        ) from exc

    question_text = question_text.strip()
    state.questions.append(question_text)
    state.conversation_history.append({"role": "assistant", "content": question_text})
    logger.info("Question %d generated, length=%d chars", question_number, len(question_text))
    return question_text


def _classify_response(state: InterviewState, response_text: str) -> str:
    """Classify a candidate's response as 'clarifying' or 'answer'."""
    current_question = state.questions[state.current_index]
    try:
        result = chat(
            _build_messages(classify_response_prompt(current_question, response_text)),
            state.provider,
            state.api_key,
            temperature=0.1,
        )
    except Exception:
        return "answer"
    return "clarifying" if "CLARIFYING" in result.strip().upper() else "answer"


def ask_clarification(state: InterviewState, question_text: str) -> tuple["InterviewState", str]:
    """
    Handle a candidate's clarifying question about the current interview question.

    Returns:
        (state, interviewer_response)
    """
    current_question = state.questions[state.current_index]
    state.conversation_history.append({"role": "user", "content": question_text})
    logger.info("Handling clarifying question for question_index=%d", state.current_index)

    try:
        response = chat(
            _build_messages(
                clarifying_question_prompt(
                    current_question,
                    question_text,
                    state.conversation_history,
                )
            ),
            state.provider,
            state.api_key,
            temperature=0.9,
        )
    except Exception as exc:
        logger.error("Clarification response failed: %s", exc)
        raise RuntimeError(
            f"Failed to respond to clarifying question via {state.provider}. "
            "Check your API key and try again."
        ) from exc

    response = response.strip()
    state.conversation_history.append({"role": "assistant", "content": response})
    logger.info("Clarification response generated, length=%d chars", len(response))
    return state, response


def handle_response(
    state: InterviewState, response_text: str
) -> tuple["InterviewState", str, str | None, str | None]:
    """
    Process a candidate's response — auto-detecting whether it is a clarifying
    question or an actual answer.

    Returns:
        (state, response_type, interviewer_reply, feedback_and_next)
        - If clarifying: response_type="clarifying", interviewer_reply=str, feedback_and_next=None
        - If answer: response_type="answer", interviewer_reply=None,
          feedback_and_next is a (feedback, next_question) tuple packed as two values
    """
    classification = _classify_response(state, response_text)

    if classification == "clarifying":
        new_state, reply = ask_clarification(state, response_text)
        return new_state, "clarifying", reply, None

    new_state, feedback, next_q = submit_answer(state, response_text)
    return new_state, "answer", feedback, next_q


def submit_answer(
    state: InterviewState, answer: str
) -> tuple["InterviewState", str, str]:
    """
    Record the candidate's answer, generate feedback, and advance the session.

    Returns:
        (state, feedback_text, next_question)
    """
    state.answers.append(answer)
    state.conversation_history.append({"role": "user", "content": answer})

    question_number = state.current_index + 1
    current_question = state.questions[state.current_index]
    logger.info("Evaluating answer for question %d, answer_length=%d chars", question_number, len(answer))

    try:
        feedback_text = chat(
            _build_messages(
                generate_feedback_prompt(current_question, answer, question_number, state.resume)
            ),
            state.provider,
            state.api_key,
            temperature=0.5,
        )
    except Exception as exc:
        logger.error("Feedback generation failed: question_number=%d error=%s", question_number, exc)
        raise RuntimeError(
            f"Failed to generate feedback for question {question_number} via {state.provider}. "
            "Check your API key and try again."
        ) from exc

    feedback_text = feedback_text.strip()
    state.feedbacks.append(feedback_text)
    score = _extract_score(feedback_text)
    state.scores.append(score)
    state.current_index += 1
    logger.info("Answer evaluated: question=%d score=%.1f/5", question_number, score)

    next_question = _generate_next_question(state)
    return state, feedback_text, next_question


def get_summary(state: InterviewState) -> str:
    """Generate and return the final interview summary report."""
    qa_pairs = [
        {"question": q, "answer": a, "feedback": f, "score": s}
        for q, a, f, s in zip(state.questions, state.answers, state.feedbacks, state.scores)
    ]
    avg_score = sum(state.scores) / len(state.scores) if state.scores else 0
    logger.info("Generating interview summary: questions_answered=%d avg_score=%.1f/5",
                len(qa_pairs), avg_score)

    try:
        summary_text = chat(
            _build_messages(generate_summary_prompt(state.jd, state.role_info, qa_pairs, state.resume)),
            state.provider,
            state.api_key,
            temperature=0.5,
        )
    except Exception as exc:
        logger.error("Summary generation failed: %s", exc)
        raise RuntimeError(
            f"Failed to generate interview summary via {state.provider}. "
            "Check your API key and try again."
        ) from exc

    logger.info("Interview summary generated, length=%d chars", len(summary_text.strip()))
    return summary_text.strip()


def get_progress(state: InterviewState) -> int:
    """Return the number of questions asked so far."""
    return len(state.questions)


In [ ]:
%%writefile app.py
import logging
import os
import gradio as gr
from dotenv import load_dotenv

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger(__name__)

load_dotenv()  # Load .env if present

# Gradio 4.x requires type="messages" for OpenAI-style chat dicts; Gradio 5.x removed the param.
_GR_MAJOR = int(gr.__version__.split(".")[0])

# Patch gradio_client schema parser to handle boolean JSON schemas (pydantic v2 compat)
try:
    import gradio_client.utils as _gcu
    _orig_json_schema = _gcu._json_schema_to_python_type

    def _patched_json_schema(schema, defs):
        if isinstance(schema, bool):
            return "Any" if schema else "None"
        return _orig_json_schema(schema, defs)

    _gcu._json_schema_to_python_type = _patched_json_schema
except Exception:
    pass

from interview_agent import start_interview, handle_response, get_progress, get_summary
from llm_client import (
    DEFAULT_PROVIDER,
    get_api_key,
    get_available_providers,
    synthesize_speech,
    transcribe_audio,
)


def make_progress_md(asked: int) -> str:
    return f"**Questions asked: {asked}**"


def _msg(role: str, content: str) -> dict:
    """Helper to build a gradio 6 messages-format chat entry."""
    return {"role": role, "content": content}


def _voice_update_for_text(text: str) -> tuple[dict, str]:
    """Generate audio update payload for interviewer text."""
    try:
        audio_path = synthesize_speech(text)
        return gr.update(value=audio_path, visible=True), ""
    except Exception as exc:
        return gr.update(value=None, visible=False), f"⚠️ Voice playback unavailable: {exc}"


with gr.Blocks(title="AI Interview Coach") as demo:
    state = gr.State(None)

    gr.Markdown("# 🤖 AI Interview Coach")
    gr.Markdown("Practice your interview skills with an AI-powered interviewer tailored to any job description.")
    gr.Markdown("<p style='text-align:right; color:gray; font-size:0.85em;'>Created by Richmond Dzoku</p>")

    error_display = gr.Markdown("", visible=False)

    with gr.Group(visible=True) as setup_panel:
        jd_input = gr.Textbox(
            label="Job Description",
            placeholder=(
                "Paste the job description here…\n\n"
                "Example: We are looking for a Senior Backend Engineer with 5+ years of "
                "experience in Python, REST APIs, and cloud infrastructure (AWS/GCP)…"
            ),
            lines=6,
        )
        resume_input = gr.Textbox(
            label="Resume (optional)",
            placeholder="Paste your resume here to get personalized feedback…",
            lines=4,
        )
        provider_dropdown = gr.Dropdown(
            choices=get_available_providers(),
            value=DEFAULT_PROVIDER,
            label="LLM Provider",
        )
        api_key_input = gr.Textbox(
            label="API Key",
            type="password",
            placeholder="Enter your API key…",
            value="",
            visible=not bool(get_api_key(DEFAULT_PROVIDER)),
        )
        api_key_status = gr.Markdown(
            "🔑 **API key loaded from environment variable.**",
            visible=bool(get_api_key(DEFAULT_PROVIDER)),
        )
        start_btn = gr.Button("🚀 Start Interview", variant="primary")
        gr.Markdown(
            "Need a free API key? Get one from "
            "[Groq](https://console.groq.com) (recommended) or "
            "[Google AI Studio](https://aistudio.google.com)",
            visible=not bool(get_api_key(DEFAULT_PROVIDER)),
        )

    # ── Provider change: update API key field dynamically ─────────────────────

    def handle_provider_change(provider):
        env_key = get_api_key(provider)
        has_key = bool(env_key)
        return (
            gr.update(visible=not has_key, value=""),
            gr.update(visible=has_key),
        )

    provider_dropdown.change(
        handle_provider_change,
        inputs=[provider_dropdown],
        outputs=[api_key_input, api_key_status],
    )

    progress_md = gr.Markdown("", visible=False)

    chatbot = gr.Chatbot(height=500, label="Interview", visible=False, **({} if _GR_MAJOR >= 5 else {"type": "messages"}))
    interviewer_audio = gr.Audio(
        label="🔊 Interviewer Voice",
        visible=False,
        autoplay=True,
    )

    with gr.Group(visible=False) as answer_panel:
        answer_input = gr.Textbox(label="Your Response", lines=4, placeholder="Type your answer or ask a clarifying question…")
        speech_input = gr.Audio(
            label="🎤 Speak Your Answer (optional)",
            sources=["microphone", "upload"],
            type="filepath",
        )
        with gr.Row():
            transcribe_btn = gr.Button("📝 Transcribe Speech", variant="secondary")
            submit_btn = gr.Button("✅ Submit", variant="primary")
            end_btn = gr.Button("⏭️ End Interview", variant="secondary")

    new_interview_btn = gr.Button("🔄 Start New Interview", visible=False)

    # ── Handlers ──────────────────────────────────────────────────────────────

    def handle_start(jd, resume, provider, api_key):
        if not jd.strip():
            logger.warning("Start interview attempted with empty job description")
            yield (gr.update(), gr.update(value="❌ Please enter a job description.", visible=True),
                   gr.update(), gr.update(), gr.update(), gr.update(), gr.update(),
                   gr.update(interactive=True, value="🚀 Start Interview"), gr.update())
            return

        api_key = (api_key or "").strip()
        resolved_key = api_key or get_api_key(provider)
        if not resolved_key:
            logger.warning("Start interview attempted without API key for provider=%s", provider)
            yield (gr.update(), gr.update(value="❌ Please enter your API key.", visible=True),
                   gr.update(), gr.update(), gr.update(), gr.update(), gr.update(),
                   gr.update(interactive=True, value="🚀 Start Interview"), gr.update())
            return
        yield (gr.update(), gr.update(value="", visible=False),
               gr.update(), gr.update(), gr.update(), gr.update(), gr.update(),
               gr.update(interactive=False, value="⏳ Starting…"),
               gr.update(value=None, visible=False))

        logger.info("Starting interview: provider=%s jd_length=%d resume_length=%d",
                     provider, len(jd), len((resume or "")))
        try:
            new_state, first_question = start_interview(jd, provider, resolved_key, resume=(resume or "").strip())
        except Exception as exc:
            logger.error("Interview start failed: %s", exc)
            yield (gr.update(), gr.update(value=f"❌ {exc}", visible=True),
                   gr.update(visible=True), gr.update(visible=False),
                   gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
                   gr.update(interactive=True, value="🚀 Start Interview"),
                   gr.update(value=None, visible=False))
            return

        asked = get_progress(new_state)
        history = [_msg("assistant", first_question)]
        voice_update, voice_warning = _voice_update_for_text(first_question)
        error_update = (
            gr.update(value=voice_warning, visible=True)
            if voice_warning
            else gr.update(value="", visible=False)
        )

        yield (new_state, error_update,
               gr.update(visible=False),
               gr.update(value=make_progress_md(asked), visible=True),
               gr.update(value=history, visible=True),
               gr.update(visible=True), gr.update(visible=False),
               gr.update(interactive=True, value="🚀 Start Interview"),
               voice_update)

    start_btn.click(
        handle_start,
        inputs=[jd_input, resume_input, provider_dropdown, api_key_input],
        outputs=[state, error_display, setup_panel, progress_md, chatbot, answer_panel, new_interview_btn, start_btn, interviewer_audio],
    )

    def handle_transcribe(audio_path, provider, api_key):
        if not audio_path:
            logger.warning("Transcribe attempted with no audio file")
            return (
                gr.update(),
                gr.update(value="❌ Please record or upload audio before transcribing.", visible=True),
            )

        resolved_key = (api_key or "").strip() or get_api_key(provider)
        if not resolved_key:
            return (
                gr.update(),
                gr.update(value="❌ Please enter your API key before transcribing.", visible=True),
            )

        logger.info("Transcribing audio: provider=%s", provider)
        try:
            transcript = transcribe_audio(audio_path, provider, resolved_key)
        except Exception as exc:
            logger.error("Transcription failed in UI handler: %s", exc)
            return (
                gr.update(),
                gr.update(value=f"❌ Speech transcription failed: {exc}", visible=True),
            )

        logger.info("Transcription successful in UI handler")
        return (
            gr.update(value=transcript),
            gr.update(value="", visible=False),
        )

    transcribe_btn.click(
        handle_transcribe,
        inputs=[speech_input, provider_dropdown, api_key_input],
        outputs=[answer_input, error_display],
    )

    def handle_submit(current_state, answer, chat_history):
        if not answer.strip():
            logger.warning("Submit attempted with empty response")
            yield (current_state, gr.update(value="❌ Please type a response before submitting.", visible=True),
                   chat_history, gr.update(), gr.update(), gr.update(), answer, gr.update(), gr.update())
            return

        logger.info("Processing candidate response, length=%d chars", len(answer.strip()))
        try:
            new_state, response_type, reply, next_q = handle_response(current_state, answer)
        except Exception as exc:
            logger.error("Response handling failed: %s", exc)
            yield (current_state, gr.update(value=f"❌ {exc}", visible=True),
                   chat_history, gr.update(), gr.update(), gr.update(), answer, gr.update(), gr.update())
            return

        history = list(chat_history)
        history.append(_msg("user", answer))
        logger.info("Response processed: type=%s", response_type)

        if response_type == "clarifying":
            history.append(_msg("assistant", reply))
            voice_update, voice_warning = _voice_update_for_text(reply)
            error_update = (
                gr.update(value=voice_warning, visible=True)
                if voice_warning
                else gr.update(value="", visible=False)
            )
            yield (new_state, error_update, history,
                   gr.update(), gr.update(visible=True), gr.update(visible=False),
                   "", None, voice_update)
        else:
            history.append(_msg("assistant", reply))
            history.append(_msg("assistant", next_q))
            asked = get_progress(new_state)
            voice_text = f"{reply}\n\n{next_q}"
            voice_update, voice_warning = _voice_update_for_text(voice_text)
            error_update = (
                gr.update(value=voice_warning, visible=True)
                if voice_warning
                else gr.update(value="", visible=False)
            )
            yield (new_state, error_update, history,
                   gr.update(value=make_progress_md(asked), visible=True),
                   gr.update(visible=True), gr.update(visible=False), "", None, voice_update)

    submit_btn.click(
        handle_submit,
        inputs=[state, answer_input, chatbot],
        outputs=[state, error_display, chatbot, progress_md, answer_panel, new_interview_btn, answer_input, speech_input, interviewer_audio],
    )

    def handle_end(current_state, chat_history):
        if current_state is None:
            logger.warning("End interview attempted with no active session")
            return (current_state, gr.update(value="❌ No active interview session.", visible=True),
                    chat_history, gr.update(), gr.update(), gr.update())
        logger.info("Ending interview: questions_answered=%d", len(current_state.answers))
        try:
            summary = get_summary(current_state)
        except Exception as exc:
            logger.error("Summary generation failed in UI handler: %s", exc)
            return (current_state, gr.update(value=f"❌ {exc}", visible=True),
                    chat_history, gr.update(), gr.update(), gr.update())

        history = list(chat_history)
        history.append(_msg("assistant", summary))
        voice_update, voice_warning = _voice_update_for_text(summary)
        error_update = (
            gr.update(value=voice_warning, visible=True)
            if voice_warning
            else gr.update(value="", visible=False)
        )
        return (current_state, error_update, history,
                gr.update(visible=False), gr.update(visible=True), voice_update)

    end_btn.click(
        handle_end,
        inputs=[state, chatbot],
        outputs=[state, error_display, chatbot, answer_panel, new_interview_btn, interviewer_audio],
    )

    def handle_new_interview():
        logger.info("Starting new interview session (reset)")
        return (None, gr.update(value="", visible=False), gr.update(visible=True),
                gr.update(value="", visible=False), gr.update(value=[], visible=False),
                gr.update(visible=False), gr.update(visible=False), "", "", "", "", None, gr.update(value=None, visible=False))

    new_interview_btn.click(
        handle_new_interview,
        inputs=[],
        outputs=[state, error_display, setup_panel, progress_md, chatbot,
                 answer_panel, new_interview_btn, jd_input, resume_input, api_key_input, answer_input, speech_input, interviewer_audio],
    )

if __name__ == "__main__":
    server_name = "0.0.0.0" if os.getenv("SPACE_ID") or os.getenv("FLY_APP_NAME") else "127.0.0.1"
    demo.launch(server_name=server_name)


## 4 · Launch the app 🚀

This starts the Gradio server and prints a **public URL** (valid for 72 hours).
Share the link with anyone — no Colab access required on their end.

⚠️ **Keep this cell running** — closing it stops the app.


In [ ]:
import subprocess, sys, os

# Patch the launch call to use share=True for a public Colab link
os.environ['GRADIO_SERVER_NAME'] = '0.0.0.0'

# Import and launch with share=True for public URL
from app import demo
demo.launch(share=True, server_name='0.0.0.0')
